# Annotation agreement analysis

This notebook loads the original zero-shot outputs and the human annotation files, then compares videos, emotions, and thematic categories. The final section focuses on no-overlap thematic cases for `Expression of personal feelings` and `Comparison`.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score

responses_dir = Path("Data") / "Anotation files" / "Responses"

original = pd.read_csv(responses_dir / "original_comments.csv")
comments_1 = pd.read_excel(responses_dir / "comments_1.xlsx")
comments_2 = pd.read_excel(responses_dir / "comments_2.xlsx")

original_videos = pd.read_excel(responses_dir / "original_videos.xlsx")
original_videos["etiqueta"] = original_videos["etiqueta"].str.strip()
videos_1 = pd.read_excel(responses_dir / "videos_1.xlsx")
videos_2 = pd.read_excel(responses_dir / "videos_2.xlsx")

## Prepare comparison

The human thematic labels are stored from the fifth column onward. A value of `x` means that the annotator selected that label.

In [2]:
emotion_map = {
    "Positivo": "Positive",
    "Negativo": "Negative",
    "Neagtive": "Negative",
}

label_cols = comments_1.columns[4:]

comments_1_labels = comments_1[label_cols].eq("x")
comments_2_labels = comments_2[label_cols].eq("x")

comparison = pd.DataFrame({
    "comment": original["comment"],
    "emotion_original": original["emotion"].str.strip().replace(emotion_map),
    "emotion_comments_1": comments_1["emotion"].str.strip().replace(emotion_map),
    "emotion_comments_2": comments_2["emotion"].str.strip().replace(emotion_map),
    "label_original": original["etiqueta"],
    "labels_comments_1": comments_1_labels.apply(lambda row: ", ".join(row.index[row]), axis=1),
    "labels_comments_2": comments_2_labels.apply(lambda row: ", ".join(row.index[row]), axis=1),
})

comparison["emotion_match_comments_1"] = comparison["emotion_original"].eq(comparison["emotion_comments_1"])
comparison["emotion_match_comments_2"] = comparison["emotion_original"].eq(comparison["emotion_comments_2"])

comparison["zero_shot_label_selected_by_annotator_1"] = [
    comments_1_labels.loc[i, label]
    for i, label in original["etiqueta"].items()
]

comparison["zero_shot_label_selected_by_annotator_2"] = [
    comments_2_labels.loc[i, label]
    for i, label in original["etiqueta"].items()
]

non_informative_texts = [
    "@Felix Calderón",
    "@@lewismanzon5074",
]

comparison["non_informative"] = comparison["comment"].str.strip().isin(non_informative_texts)
comparison_valid = comparison[~comparison["non_informative"]].copy()

non_informative_comments = comparison[comparison["non_informative"]][[
    "comment",
    "emotion_original",
    "emotion_comments_1",
    "emotion_comments_2",
    "label_original",
    "labels_comments_1",
    "labels_comments_2",
]]

comparison_valid.tail()

,comment,emotion_original,emotion_comments_1,emotion_comments_2,label_original,labels_comments_1,labels_comments_2,emotion_match_comments_1,emotion_match_comments_2,zero_shot_label_selected_by_annotator_1,zero_shot_label_selected_by_annotator_2,non_informative
95,Amo os vídeos da camila ❤\nBeijos do Brasil,Positive,Positive,Positive,Expression of personal feelings,"Compliment, Expression of personal feelings, G...","Compliment, Expression of personal feelings, G...",True,True,True,True,False
96,"Go back to the garage, that car isn’t going to...",Negative,Negative,Negative,Comparison,"Advice Give, Criticism","Advice Give, Criticism",True,True,False,False,False
97,Gracias!!!! me quedo una duda no se si me la p...,Negative,Positive,Positive,Expression of personal feelings,"Advice request, Thanking","Medical treatment, Thanking",False,False,False,False,False
98,muy bueno.. lo unico que me parecio muy molest...,Positive,Positive,Positive,Comparison,"Compliment, Criticism","Compliment, Criticism",True,True,False,False,False
99,Información muy valiosa 👍,Positive,Positive,Positive,Advice request,Compliment,Compliment,True,True,False,False,False


## Non-informative comments

Comments excluded from the comment-level metrics because they contain only a username.

In [3]:
non_informative_comments

,comment,emotion_original,emotion_comments_1,emotion_comments_2,label_original,labels_comments_1,labels_comments_2
43,@Felix Calderón,Positive,Neutral,Neutral,Criticism,Other,
69,@@lewismanzon5074,Positive,Neutral,Neutral,Comparison,Other,


## Videos

In [4]:
video_labels_original = original_videos["etiqueta"].str.strip()
video_labels_1 = videos_1["etiqueta"].str.strip()
video_labels_2 = videos_2["etiqueta"].str.strip()

video_interrater_kappa = pd.DataFrame([{
    "comparison": "videos_1 vs videos_2",
    "n_videos": len(video_labels_original),
    "cohen_kappa": cohen_kappa_score(video_labels_1, video_labels_2),
}])

video_annotator_metrics = pd.DataFrame({
    "annotator": ["videos_1", "videos_2"],
    "n_videos": [len(video_labels_original), len(video_labels_original)],
    "accuracy_original_vs_annotator": [
        accuracy_score(video_labels_original, video_labels_1),
        accuracy_score(video_labels_original, video_labels_2),
    ],
    "macro_f1_original_vs_annotator": [
        f1_score(video_labels_original, video_labels_1, average="macro", zero_division=0),
        f1_score(video_labels_original, video_labels_2, average="macro", zero_division=0),
    ],
})

relevant_video_labels = ["Scientific", "Pseudoscientific"]
video_confusion_labels = ["Scientific", "Pseudoscientific", "Irrelevant"]
relevant_video_mask = video_labels_original.isin(relevant_video_labels)

video_relevant_comparison = pd.concat([
    pd.DataFrame({
        "annotator": "videos_1",
        "original_label": video_labels_original[relevant_video_mask],
        "annotator_label": video_labels_1[relevant_video_mask],
    }),
    pd.DataFrame({
        "annotator": "videos_2",
        "original_label": video_labels_original[relevant_video_mask],
        "annotator_label": video_labels_2[relevant_video_mask],
    }),
], ignore_index=True)

video_relevant_comparison["correct"] = video_relevant_comparison["original_label"].eq(
    video_relevant_comparison["annotator_label"]
)

video_relevant_accuracy_by_class = (
    video_relevant_comparison
    .groupby(["annotator", "original_label"])
    .agg(n_cases=("correct", "size"), correct=("correct", "sum"))
    .reset_index()
)
video_relevant_accuracy_by_class["percent_correct"] = (
    video_relevant_accuracy_by_class["correct"]
    / video_relevant_accuracy_by_class["n_cases"]
    * 100
)

video_confusion_videos_1 = (
    pd.crosstab(
        video_labels_original,
        video_labels_1,
        rownames=["original"],
        colnames=["videos_1"],
    )
    .reindex(index=video_confusion_labels, columns=video_confusion_labels, fill_value=0)
)

video_confusion_videos_2 = (
    pd.crosstab(
        video_labels_original,
        video_labels_2,
        rownames=["original"],
        colnames=["videos_2"],
    )
    .reindex(index=video_confusion_labels, columns=video_confusion_labels, fill_value=0)
)

video_combined_confusion_matrix = pd.concat({
    "Annotator 1": video_confusion_videos_1,
    "Annotator 2": video_confusion_videos_2,
}, axis=1)
video_combined_confusion_matrix[("Total", "n")] = video_confusion_videos_1.sum(axis=1)
video_combined_confusion_matrix.index.name = "Original category"
video_combined_confusion_matrix.columns.names = ["Annotator", "Assigned category"]

video_combined_confusion_matrix_style = video_combined_confusion_matrix.style
for label in video_confusion_labels:
    video_combined_confusion_matrix_style = (
        video_combined_confusion_matrix_style
        .set_properties(
            subset=([label], [("Annotator 1", label), ("Annotator 2", label)]),
            **{"background-color": "#fff2cc", "font-weight": "bold"},
        )
    )

video_category_agreement = pd.DataFrame({
    "original_category": video_confusion_labels,
    "n": video_confusion_videos_1.sum(axis=1).to_numpy(),
    "annotator_1_same_n": [
        video_confusion_videos_1.loc[label, label] for label in video_confusion_labels
    ],
    "annotator_2_same_n": [
        video_confusion_videos_2.loc[label, label] for label in video_confusion_labels
    ],
})
video_category_agreement["annotator_1_same_percent"] = (
    video_category_agreement["annotator_1_same_n"]
    / video_category_agreement["n"]
    * 100
).round(1)
video_category_agreement["annotator_2_same_percent"] = (
    video_category_agreement["annotator_2_same_n"]
    / video_category_agreement["n"]
    * 100
).round(1)
video_category_agreement = video_category_agreement[[
    "original_category",
    "n",
    "annotator_1_same_n",
    "annotator_1_same_percent",
    "annotator_2_same_n",
    "annotator_2_same_percent",
]]

video_category_agreement = pd.concat([
    video_category_agreement,
    pd.DataFrame([{
        "original_category": "Total",
        "n": video_category_agreement["n"].sum(),
        "annotator_1_same_n": video_category_agreement["annotator_1_same_n"].sum(),
        "annotator_1_same_percent": round(
            video_category_agreement["annotator_1_same_n"].sum()
            / video_category_agreement["n"].sum()
            * 100,
            1,
        ),
        "annotator_2_same_n": video_category_agreement["annotator_2_same_n"].sum(),
        "annotator_2_same_percent": round(
            video_category_agreement["annotator_2_same_n"].sum()
            / video_category_agreement["n"].sum()
            * 100,
            1,
        ),
    }]),
], ignore_index=True)

display(video_interrater_kappa)
display(video_annotator_metrics)
display(video_relevant_accuracy_by_class)
display(video_category_agreement)
display(video_combined_confusion_matrix_style)


,comparison,n_videos,cohen_kappa
0,videos_1 vs videos_2,30,0.829545


,annotator,n_videos,accuracy_original_vs_annotator,macro_f1_original_vs_annotator
0,videos_1,30,0.866667,0.726284
1,videos_2,30,0.766667,0.646825


,annotator,original_label,n_cases,correct,percent_correct
0,videos_1,Pseudoscientific,15,14,93.333333
1,videos_1,Scientific,11,11,100.000000
2,videos_2,Pseudoscientific,15,12,80.000000
3,videos_2,Scientific,11,10,90.909091


,original_category,n,annotator_1_same_n,annotator_1_same_percent,annotator_2_same_n,annotator_2_same_percent
0,Scientific,11,11,100.0,10,90.9
1,Pseudoscientific,15,14,93.3,12,80.0
2,Irrelevant,4,1,25.0,1,25.0
3,Total,30,26,86.7,23,76.7


## Emotions

In [5]:
emotion_summary = pd.Series({
    "match_comments_1": comparison_valid["emotion_match_comments_1"].mean(),
    "match_comments_2": comparison_valid["emotion_match_comments_2"].mean(),
    "match_both_humans": (
        comparison_valid["emotion_match_comments_1"]
        & comparison_valid["emotion_match_comments_2"]
    ).mean(),
    "match_any_human": (
        comparison_valid["emotion_match_comments_1"]
        | comparison_valid["emotion_match_comments_2"]
    ).mean(),
})

emotion_summary

match_comments_1     0.683673
match_comments_2     0.602041
match_both_humans    0.581633
match_any_human      0.704082
dtype: float64

## Metrics: emotions

In [6]:
emotion_human_agreement = pd.Series({
    "kappa": cohen_kappa_score(comparison_valid["emotion_comments_1"], comparison_valid["emotion_comments_2"]),
    "accuracy": accuracy_score(comparison_valid["emotion_comments_1"], comparison_valid["emotion_comments_2"]),
})

emotion_human_agreement

kappa       0.752066
accuracy    0.846939
dtype: float64

In [7]:
emotion_model_metrics = pd.DataFrame({
    "comparison": ["original vs comments_1", "original vs comments_2"],
    "accuracy": [
        accuracy_score(comparison_valid["emotion_original"], comparison_valid["emotion_comments_1"]),
        accuracy_score(comparison_valid["emotion_original"], comparison_valid["emotion_comments_2"]),
    ],
    "macro_f1": [
        f1_score(comparison_valid["emotion_original"], comparison_valid["emotion_comments_1"], average="macro"),
        f1_score(comparison_valid["emotion_original"], comparison_valid["emotion_comments_2"], average="macro"),
    ],
})

emotion_model_metrics

,comparison,accuracy,macro_f1
0,original vs comments_1,0.683673,0.603214
1,original vs comments_2,0.602041,0.516809


In [8]:
emotion_confusion_rows = list(pd.unique(comparison_valid["emotion_original"].dropna()))
emotion_confusion_columns = list(pd.unique(pd.concat([
    comparison_valid["emotion_original"],
    comparison_valid["emotion_comments_1"],
    comparison_valid["emotion_comments_2"],
]).dropna()))

emotion_confusion_comments_1 = (
    pd.crosstab(
        comparison_valid["emotion_original"],
        comparison_valid["emotion_comments_1"],
        rownames=["original"],
        colnames=["comments_1"],
    )
    .reindex(index=emotion_confusion_rows, columns=emotion_confusion_columns, fill_value=0)
)

emotion_confusion_comments_2 = (
    pd.crosstab(
        comparison_valid["emotion_original"],
        comparison_valid["emotion_comments_2"],
        rownames=["original"],
        colnames=["comments_2"],
    )
    .reindex(index=emotion_confusion_rows, columns=emotion_confusion_columns, fill_value=0)
)

emotion_combined_confusion_matrix = pd.concat({
    "Annotator 1": emotion_confusion_comments_1,
    "Annotator 2": emotion_confusion_comments_2,
}, axis=1)
emotion_combined_confusion_matrix[("Total", "n")] = emotion_confusion_comments_1.sum(axis=1)
emotion_combined_confusion_matrix.index.name = "Zero-shot emotion"
emotion_combined_confusion_matrix.columns.names = ["Annotator", "Assigned emotion"]

emotion_combined_confusion_matrix_style = emotion_combined_confusion_matrix.style
for label in emotion_confusion_rows:
    emotion_combined_confusion_matrix_style = (
        emotion_combined_confusion_matrix_style
        .set_properties(
            subset=([label], [("Annotator 1", label), ("Annotator 2", label)]),
            **{"background-color": "#fff2cc", "font-weight": "bold"},
        )
    )

emotion_combined_confusion_matrix_style

## Thematic categories

The zero-shot model assigns one category per comment, while human annotators can select multiple categories. Agreement is measured by checking whether the zero-shot category is included among the categories selected by each annotator.

In [9]:
comments_1_labels_valid = comments_1_labels.loc[comparison_valid.index]
comments_2_labels_valid = comments_2_labels.loc[comparison_valid.index]

zero_shot_selected_by_annotator_1 = comparison_valid["zero_shot_label_selected_by_annotator_1"]
zero_shot_selected_by_annotator_2 = comparison_valid["zero_shot_label_selected_by_annotator_2"]
zero_shot_selected_by_either_annotator = zero_shot_selected_by_annotator_1 | zero_shot_selected_by_annotator_2
zero_shot_selected_by_both_annotators = zero_shot_selected_by_annotator_1 & zero_shot_selected_by_annotator_2
weighted_human_selection_score = (
    zero_shot_selected_by_annotator_1.astype(int)
    + zero_shot_selected_by_annotator_2.astype(int)
) / 2

n_label_comments = len(comparison_valid)

zero_shot_selection_summary = pd.DataFrame({
    "comparison": [
        "Zero-shot category selected by Annotator 1",
        "Zero-shot category selected by Annotator 2",
        "Zero-shot category selected by either annotator",
        "Zero-shot category selected by both annotators",
    ],
    "n_comments": n_label_comments,
    "comments_with_selection": [
        zero_shot_selected_by_annotator_1.sum(),
        zero_shot_selected_by_annotator_2.sum(),
        zero_shot_selected_by_either_annotator.sum(),
        zero_shot_selected_by_both_annotators.sum(),
    ],
})
zero_shot_selection_summary["percent"] = (
    zero_shot_selection_summary["comments_with_selection"]
    / zero_shot_selection_summary["n_comments"]
    * 100
).round(1)

weighted_human_selection_summary = pd.DataFrame([{
    "metric": "Weighted human selection score",
    "n_comments": n_label_comments,
    "mean_score": round(weighted_human_selection_score.mean(), 3),
    "percent": round(weighted_human_selection_score.mean() * 100, 1),
}])

display(zero_shot_selection_summary)
weighted_human_selection_summary

,comparison,n_comments,comments_with_selection,percent
0,Zero-shot category selected by Annotator 1,98,48,49.0
1,Zero-shot category selected by Annotator 2,98,35,35.7
2,Zero-shot category selected by either annotator,98,52,53.1
3,Zero-shot category selected by both annotators,98,31,31.6


,metric,n_comments,mean_score,percent
0,Weighted human selection score,98,0.423,42.3


The weighted human selection score is the average number of annotators who selected the zero-shot category, divided by two: 0 when neither annotator selected it, 0.5 when one annotator selected it, and 1 when both annotators selected it.

In [10]:
label_agreement_by_comment = comparison_valid[["comment", "label_original"]].copy()
label_agreement_by_comment["annotator_1_selection"] = zero_shot_selected_by_annotator_1
label_agreement_by_comment["annotator_2_selection"] = zero_shot_selected_by_annotator_2
label_agreement_by_comment["weighted_human_selection_score"] = weighted_human_selection_score

label_agreement_by_category = (
    label_agreement_by_comment
    .groupby("label_original", dropna=False)
    .agg(
        n=("comment", "size"),
        annotator_1_selections=("annotator_1_selection", "sum"),
        annotator_2_selections=("annotator_2_selection", "sum"),
        weighted_human_selection_score=("weighted_human_selection_score", "mean"),
    )
    .reset_index()
    .rename(columns={"label_original": "Zero-shot category"})
)

label_agreement_by_category["annotator_1_selection_percent"] = (
    label_agreement_by_category["annotator_1_selections"]
    / label_agreement_by_category["n"]
    * 100
).round(1)
label_agreement_by_category["annotator_2_selection_percent"] = (
    label_agreement_by_category["annotator_2_selections"]
    / label_agreement_by_category["n"]
    * 100
).round(1)
label_agreement_by_category["weighted_human_selection_percent"] = (
    label_agreement_by_category["weighted_human_selection_score"]
    * 100
).round(1)

label_agreement_by_category = label_agreement_by_category[[
    "Zero-shot category",
    "n",
    "annotator_1_selections",
    "annotator_1_selection_percent",
    "annotator_2_selections",
    "annotator_2_selection_percent",
    "weighted_human_selection_percent",
]]

label_agreement_by_category.sort_values(
    ["weighted_human_selection_percent", "n"],
    ascending=[False, False],
)

,Zero-shot category,n,annotator_1_selections,annotator_1_selection_percent,annotator_2_selections,annotator_2_selection_percent,weighted_human_selection_percent
10,Thanking,7,7,100.0,7,100.0,100.0
6,Greetings,2,2,100.0,2,100.0,100.0
8,Medical treatment,3,2,66.7,3,100.0,83.3
0,Advice Give,1,1,100.0,0,0.0,50.0
5,Expression of personal feelings,40,20,50.0,16,40.0,45.0
3,Compliment,10,4,40.0,5,50.0,45.0
2,Comparison,23,10,43.5,2,8.7,26.1
9,Speculation,2,1,50.0,0,0.0,25.0
1,Advice request,3,1,33.3,0,0.0,16.7
4,Desires,4,0,0.0,0,0.0,0.0


## No-overlap thematic analysis

This final section focuses on comments where neither human annotator selected the category assigned by the zero-shot classifier.

In [11]:
human_common_labels = comments_1_labels_valid & comments_2_labels_valid
human_complete_agreement = comments_1_labels_valid.eq(comments_2_labels_valid).all(axis=1)
human_partial_agreement = human_common_labels.sum(axis=1).gt(0) & ~human_complete_agreement

classifier_disagreement_with_both_humans = ~zero_shot_selected_by_either_annotator
human_total_disagreement = (
    classifier_disagreement_with_both_humans
    & ~human_complete_agreement
    & ~human_partial_agreement
)

no_overlap_all_categories_base = comparison_valid[["comment", "label_original"]].copy()
no_overlap_all_categories_base["classifier_disagreement_with_both_humans"] = classifier_disagreement_with_both_humans
no_overlap_all_categories_base["human_complete_agreement"] = classifier_disagreement_with_both_humans & human_complete_agreement
no_overlap_all_categories_base["human_partial_agreement"] = classifier_disagreement_with_both_humans & human_partial_agreement
no_overlap_all_categories_base["human_total_disagreement"] = human_total_disagreement

classifier_disagreement_by_category = (
    no_overlap_all_categories_base
    .groupby("label_original", dropna=False)
    .agg(
        total_zero_shot_cases=("label_original", "size"),
        classifier_disagreement_with_both_humans=("classifier_disagreement_with_both_humans", "sum"),
    )
    .reset_index()
    .rename(columns={"label_original": "Zero-shot category"})
)
classifier_disagreement_by_category["percent_of_zero_shot_category"] = (
    classifier_disagreement_by_category["classifier_disagreement_with_both_humans"]
    / classifier_disagreement_by_category["total_zero_shot_cases"]
    * 100
).round(1)

classifier_disagreement_by_category = classifier_disagreement_by_category.sort_values(
    "classifier_disagreement_with_both_humans",
    ascending=False,
)

selected_categories = classifier_disagreement_by_category.head(2)["Zero-shot category"].tolist()

classifier_disagreement_by_category

,Zero-shot category,total_zero_shot_cases,classifier_disagreement_with_both_humans,percent_of_zero_shot_category
5,Expression of personal feelings,40,18,45.0
2,Comparison,23,13,56.5
3,Compliment,10,5,50.0
4,Desires,4,4,100.0
7,Insult,3,3,100.0
1,Advice request,3,2,66.7
9,Speculation,2,1,50.0
0,Advice Give,1,0,0.0
6,Greetings,2,0,0.0
8,Medical treatment,3,0,0.0


## Human agreement inside the two most common disagreement categories

This table keeps the two zero-shot categories with the most classifier disagreement with both humans. Complete agreement means both annotators selected the same full set of labels. Partial agreement means they shared at least one label, but their full selections were not identical. Total disagreement means they shared no label.

In [12]:
top_disagreement_categories_base = no_overlap_all_categories_base[
    no_overlap_all_categories_base["label_original"].isin(selected_categories)
    & no_overlap_all_categories_base["classifier_disagreement_with_both_humans"]
].copy()

top_disagreement_human_agreement_table = (
    top_disagreement_categories_base
    .groupby("label_original", dropna=False)
    .agg(
        classifier_disagreement_cases=("classifier_disagreement_with_both_humans", "sum"),
        complete_human_agreement_cases=("human_complete_agreement", "sum"),
        partial_human_agreement_cases=("human_partial_agreement", "sum"),
        total_human_disagreement_cases=("human_total_disagreement", "sum"),
    )
    .reset_index()
    .rename(columns={"label_original": "Zero-shot category"})
)

top_disagreement_human_agreement_table["complete_human_agreement_percent"] = (
    top_disagreement_human_agreement_table["complete_human_agreement_cases"]
    / top_disagreement_human_agreement_table["classifier_disagreement_cases"]
    * 100
).round(1)
top_disagreement_human_agreement_table["partial_human_agreement_percent"] = (
    top_disagreement_human_agreement_table["partial_human_agreement_cases"]
    / top_disagreement_human_agreement_table["classifier_disagreement_cases"]
    * 100
).round(1)
top_disagreement_human_agreement_table["total_human_disagreement_percent"] = (
    top_disagreement_human_agreement_table["total_human_disagreement_cases"]
    / top_disagreement_human_agreement_table["classifier_disagreement_cases"]
    * 100
).round(1)

top_disagreement_human_agreement_table["category_order"] = top_disagreement_human_agreement_table["Zero-shot category"].map(
    {category: order for order, category in enumerate(selected_categories)}
)
top_disagreement_human_agreement_table = (
    top_disagreement_human_agreement_table
    .sort_values("category_order")
    [[
        "Zero-shot category",
        "classifier_disagreement_cases",
        "complete_human_agreement_cases",
        "complete_human_agreement_percent",
        "partial_human_agreement_cases",
        "partial_human_agreement_percent",
        "total_human_disagreement_cases",
        "total_human_disagreement_percent",
    ]]
)

top_disagreement_human_agreement_table

,Zero-shot category,classifier_disagreement_cases,complete_human_agreement_cases,complete_human_agreement_percent,partial_human_agreement_cases,partial_human_agreement_percent,total_human_disagreement_cases,total_human_disagreement_percent
1,Expression of personal feelings,18,2,11.1,12,66.7,4,22.2
0,Comparison,13,4,30.8,9,69.2,0,0.0


## Shared human label coincidences

This table counts only labels selected by both annotators for the same comment within the no-overlap cases for the two most common disagreement categories. A shared label contributes one coincidence, not two selections. Percentages use the total number of shared label coincidences in each zero-shot category as the denominator.

In [14]:
shared_human_label_tables = []

for model_label in selected_categories:
    category_index = top_disagreement_categories_base[
        top_disagreement_categories_base["label_original"].eq(model_label)
    ].index

    shared_label_counts = (
        comments_1_labels_valid.loc[category_index]
        & comments_2_labels_valid.loc[category_index]
    ).sum()
    n_shared_labels = shared_label_counts.sum()

    category_table = pd.DataFrame({
        "Zero-shot category": model_label,
        "Shared human label": label_cols,
        "n_shared_label_coincidences": shared_label_counts.values,
        "total_shared_label_coincidences": n_shared_labels,
    })
    category_table = category_table[
        category_table["n_shared_label_coincidences"].gt(0)
    ].copy()
    category_table["percent_of_shared_label_coincidences"] = (
        category_table["n_shared_label_coincidences"]
        / category_table["total_shared_label_coincidences"]
        * 100
    ).round(1)
    shared_human_label_tables.append(category_table)

shared_human_labels_table = pd.concat(
    shared_human_label_tables,
    ignore_index=True,
)
shared_human_labels_table["category_order"] = shared_human_labels_table["Zero-shot category"].map(
    {category: order for order, category in enumerate(selected_categories)}
)
shared_human_labels_table = (
    shared_human_labels_table
    .sort_values(
        ["category_order", "n_shared_label_coincidences"],
        ascending=[True, False],
    )
    [[
        "Zero-shot category",
        "Shared human label",
        "n_shared_label_coincidences",
        "total_shared_label_coincidences",
        "percent_of_shared_label_coincidences",
    ]]
)

shared_human_labels_table

,Zero-shot category,Shared human label,n_shared_label_coincidences,total_shared_label_coincidences,percent_of_shared_label_coincidences
6,Expression of personal feelings,Thanking,5,23,21.7
0,Expression of personal feelings,Advice Give,4,23,17.4
4,Expression of personal feelings,Medical treatment,4,23,17.4
1,Expression of personal feelings,Compliment,3,23,13.0
2,Expression of personal feelings,Criticism,3,23,13.0
3,Expression of personal feelings,Desires,2,23,8.7
5,Expression of personal feelings,Speculation,2,23,8.7
7,Comparison,Advice Give,5,20,25.0
10,Comparison,Criticism,5,20,25.0
12,Comparison,Medical treatment,3,20,15.0
